# Qiskit Patterns y Primitivas V2

**Laboratorio 31 · Nivel muy avanzado**

Qiskit Patterns es el framework oficial para construir algoritmos cuánticos
ejecutables en hardware real (IBM Quantum). Sigue cuatro fases:

1. **Map** — traducir el problema al circuito cuántico y observable
2. **Optimize** — transpilar y optimizar para el hardware
3. **Execute** — ejecutar con primitivas (Sampler/Estimator)
4. **Post-process** — interpretar y analizar resultados

Este laboratorio usa simuladores locales (no requiere credenciales IBM).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from qiskit import QuantumCircuit, transpile
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import SparsePauliOp, Statevector

# Primitivas V2 locales (sin IBM Quantum)
from qiskit.primitives import StatevectorSampler, StatevectorEstimator

print('Qiskit primitivas V2 disponibles (modo local):')
print(f'  StatevectorSampler: {StatevectorSampler}')
print(f'  StatevectorEstimator: {StatevectorEstimator}')

## 1. El patrón Qiskit Patterns

El workflow completo para cualquier algoritmo cuántico en IBM Quantum.

In [ ]:
# Diagrama del workflow Qiskit Patterns
workflow = """
┌─────────────────────────────────────────────────────────────────┐
│                    QISKIT PATTERNS WORKFLOW                     │
├─────────────┬──────────────┬───────────────┬────────────────────┤
│   1. MAP    │ 2. OPTIMIZE  │  3. EXECUTE   │  4. POST-PROCESS   │
├─────────────┼──────────────┼───────────────┼────────────────────┤
│ Problema    │ Transpilación│ Primitiva     │ Extracción de      │
│    ↓        │    ↓         │   Sampler     │   resultados       │
│ Circuito    │ Optimización │   Estimator   │ Mitigación errores │
│    +        │ ISA circuit  │      ↓        │ Análisis estadíst. │
│ Observable  │ (hardware)   │ PrimitiveJob  │ Visualización      │
└─────────────┴──────────────┴───────────────┴────────────────────┘

Primitivas V2 (desde Qiskit 1.0):
  Sampler  → BitArray  (distribución de medidas)
  Estimator→ PubResult (valor esperado de observable)

Formato PUB (Primitive Unified Bloc):
  Sampler:   (circuit, [shots])
  Estimator: (circuit, observables, parameter_values)
"""
print(workflow)

## 2. FASE 1 — Map: problema → circuito + observable

Ejemplo: VQE para H₂ con ansatz de 2 qubits.

In [ ]:
# ── Fase 1: Map ──────────────────────────────────────────────────

# Hamiltoniano H₂ mínimo (Jordan-Wigner, 2 qubits)
# E_FCI = -1.8572 Ha (referencia exacta)
H2_hamiltonian = SparsePauliOp.from_list([
    ('II', -1.0523732),
    ('IZ',  0.3979374),
    ('ZI', -0.3979374),
    ('ZZ', -0.0112801),
    ('XX',  0.1809312),
])

print('Hamiltoniano H₂ (2 qubits):')
print(H2_hamiltonian)
print(f'\nNúmero de términos Pauli: {len(H2_hamiltonian)}')

# Ansatz UCCSD simplificado: ry + cx
theta = ParameterVector('θ', 1)

ansatz = QuantumCircuit(2)
ansatz.x(0)                 # estado HF |01⟩
ansatz.ry(theta[0], 1)
ansatz.cx(1, 0)

print('\nAnsatz UCCSD (2q):')
print(ansatz.draw())

## 3. FASE 2 — Optimize: transpilación para hardware

En modo local usamos `generate_preset_pass_manager` (ISA = Instruction Set Architecture).

In [ ]:
# ── Fase 2: Optimize ─────────────────────────────────────────────

from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_aer import AerSimulator

# Simular un backend de 127 qubits (Eagle r3) — sin credenciales
backend = AerSimulator()

# Generar pass manager con nivel de optimización 1
pm = generate_preset_pass_manager(optimization_level=1, backend=backend)

# Transpilar el ansatz al ISA del backend
ansatz_isa = pm.run(ansatz)

print('Circuito original:')
print(f'  Profundidad: {ansatz.depth()}, Puertas: {ansatz.size()}')
print('\nCircuito transpilado (ISA):')
print(f'  Profundidad: {ansatz_isa.depth()}, Puertas: {ansatz_isa.size()}')
print(f'  Operaciones: {dict(ansatz_isa.count_ops())}')

# También transpilar el observable al layout del hardware
H2_isa = H2_hamiltonian.apply_layout(ansatz_isa.layout)
print(f'\nObservable adaptado al layout: {len(H2_isa)} términos Pauli')

## 4. FASE 3 — Execute: Estimator V2

El `Estimator` calcula ⟨ψ(θ)|H|ψ(θ)⟩ directamente sin medir.

In [ ]:
# ── Fase 3: Execute con Estimator V2 ─────────────────────────────

estimator = StatevectorEstimator()

# PUB format: (circuit, observables, parameter_values)
# parameter_values debe ser array con shape (num_params,)
theta_val = np.array([0.5])  # valor de prueba

pub = (ansatz, H2_hamiltonian, theta_val)  # sin transpilar — modo local
job = estimator.run([pub])
result = job.result()

# Extraer valor esperado
pub_result = result[0]
energia = pub_result.data.evs  # expectation values

print(f'⟨H₂⟩ con θ={theta_val[0]:.4f}: E = {float(energia):.6f} Ha')
print(f'Referencia exacta (FCI): E = -1.8572 Ha')

# Estructura del resultado V2
print(f'\nTipo de resultado: {type(result)}')
print(f'Tipo PubResult:    {type(pub_result)}')
print(f'Campos de .data:   {dir(pub_result.data)}')

## 5. FASE 4 — Post-process: VQE completo con Qiskit Patterns

In [ ]:
# ── Fase 4: Post-process + VQE completo ──────────────────────────

from scipy.optimize import minimize

estimator_vqe = StatevectorEstimator()
energias_vqe = []

def cost_function(params):
    """Función de coste para VQE usando Estimator V2."""
    pub = (ansatz, H2_hamiltonian, np.array(params))
    result = estimator_vqe.run([pub]).result()
    energia = float(result[0].data.evs)
    energias_vqe.append(energia)
    return energia

# Optimización
theta0 = np.array([0.1])
resultado = minimize(cost_function, theta0, method='COBYLA',
                     options={'maxiter': 100, 'rhobeg': 0.5})

E_optimo = resultado.fun
theta_optimo = resultado.x

print('═══ Resultado VQE (Qiskit Patterns) ═══')
print(f'θ óptimo:           {theta_optimo[0]:.6f} rad')
print(f'Energía VQE:        {E_optimo:.6f} Ha')
print(f'Referencia FCI:     -1.857300 Ha')
print(f'Error:              {abs(E_optimo - (-1.8572)):.6f} Ha')
print(f'Iteraciones:        {len(energias_vqe)}')

# Gráfica de convergencia
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(energias_vqe, 'b-o', ms=3, lw=1.5, alpha=0.7)
ax.axhline(-1.8572, color='red', ls='--', lw=2, label='FCI exacto (-1.8572 Ha)')
ax.axhline(E_optimo, color='green', ls=':', lw=2, label=f'VQE óptimo ({E_optimo:.4f} Ha)')
ax.set_xlabel('Iteración')
ax.set_ylabel('Energía (Ha)')
ax.set_title('Convergencia VQE con Qiskit Patterns (Estimator V2)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Sampler V2: medidas y distribuciones de probabilidad

El `Sampler` devuelve un `BitArray` — la representación compacta de medidas.

In [ ]:
# ── Sampler V2: BitArray y distribuciones ─────────────────────────

sampler = StatevectorSampler()

# Circuito de Bell con medida
qc_bell = QuantumCircuit(2)
qc_bell.h(0)
qc_bell.cx(0, 1)
qc_bell.measure_all()

# PUB format para Sampler: (circuit, shots)
shots = 10_000
pub_bell = (qc_bell,)
job_s = sampler.run([pub_bell], shots=shots)
result_s = job_s.result()

# Resultado es un DataBin con campo 'meas' (BitArray)
pub_res = result_s[0]
bit_array = pub_res.data.meas  # BitArray

print(f'Tipo de resultado:  {type(bit_array)}')
print(f'Shape del BitArray: {bit_array.shape}')
print(f'Número de bits:     {bit_array.num_bits}')
print(f'Número de shots:    {bit_array.num_shots}')

# Convertir a counts
counts = bit_array.get_counts()
print(f'\nCounts (top 5):')
for k, v in sorted(counts.items(), key=lambda x: -x[1])[:5]:
    pct = 100 * v / shots
    print(f'  |{k}⟩: {v:>6} ({pct:.1f}%)')

# Convertir a entero para análisis estadístico
int_array = bit_array.get_int_counts()
print(f'\nInt counts: {int_array}')

In [ ]:
# Comparar múltiples circuitos en un solo job (batching de PUBs)

def circuito_ghz(n: int) -> QuantumCircuit:
    qc = QuantumCircuit(n)
    qc.h(0)
    for i in range(n - 1):
        qc.cx(i, i + 1)
    qc.measure_all()
    return qc

sampler_batch = StatevectorSampler()
shots_batch = 5_000

# Batch de PUBs: 3, 4, 5 qubits GHZ en paralelo
pubs_batch = [(circuito_ghz(n),) for n in [2, 3, 4, 5]]
job_batch = sampler_batch.run(pubs_batch, shots=shots_batch)
results_batch = job_batch.result()

print('Batch de circuitos GHZ (n=2..5):')
print(f'{"n":>3} | {"Estado dominante":>18} | {"P(estado)"}:>10')
print('-' * 45)

for i, n in enumerate([2, 3, 4, 5]):
    counts_i = results_batch[i].data.meas.get_counts()
    top = max(counts_i, key=counts_i.get)
    p_top = counts_i[top] / shots_batch
    print(f'{n:>3} | {"|"+top+"⟩":>18} | {p_top:.3f}')
    
print('\n(GHZ perfecto: solo |0...0⟩ y |1...1⟩ con prob ~0.5 cada uno)')

## 7. Primitivas con parámetros vectorizados

V2 permite pasar múltiples valores de parámetros en un solo PUB (batching interno).

In [ ]:
# ── Vectorización de parámetros en Estimator V2 ───────────────────

# Circuito con 2 parámetros
theta_vec = ParameterVector('θ', 2)

qc_2param = QuantumCircuit(2)
qc_2param.ry(theta_vec[0], 0)
qc_2param.ry(theta_vec[1], 1)
qc_2param.cx(0, 1)

# Observable: correlación ZZ
obs_zz = SparsePauliOp.from_list([('ZZ', 1.0)])

# Batch de valores de parámetros: shape (N, num_params)
n_vals = 50
t1_vals = np.linspace(0, 2 * np.pi, n_vals)
t2_vals = np.linspace(0, np.pi, n_vals)

# param_values shape: (n_vals, 2) — todos los puntos en un solo PUB
param_values = np.column_stack([t1_vals, t2_vals])

estimator_vec = StatevectorEstimator()
pub_vec = (qc_2param, obs_zz, param_values)
result_vec = estimator_vec.run([pub_vec]).result()

# evs tiene shape (n_vals,)
zz_vals = result_vec[0].data.evs
print(f'Shape de evs: {zz_vals.shape}')
print(f'<ZZ> mín: {zz_vals.min():.4f}, máx: {zz_vals.max():.4f}')

# Landscape 2D
n_grid = 20
t1g = np.linspace(0, 2 * np.pi, n_grid)
t2g = np.linspace(0, 2 * np.pi, n_grid)
T1, T2 = np.meshgrid(t1g, t2g)
param_grid = np.column_stack([T1.ravel(), T2.ravel()])

pub_grid = (qc_2param, obs_zz, param_grid)
result_grid = estimator_vec.run([pub_grid]).result()
ZZ_grid = result_grid[0].data.evs.reshape(n_grid, n_grid)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(t1_vals, zz_vals, 'b-', lw=2)
axes[0].set_xlabel('θ₁ (rad)')
axes[0].set_ylabel('⟨ZZ⟩')
axes[0].set_title('Correlación ZZ vs θ₁ (θ₂=linspace)')
axes[0].grid(alpha=0.3)

im = axes[1].contourf(T1, T2, ZZ_grid, levels=20, cmap='RdBu_r')
plt.colorbar(im, ax=axes[1], label='⟨ZZ⟩')
axes[1].set_xlabel('θ₁ (rad)')
axes[1].set_ylabel('θ₂ (rad)')
axes[1].set_title('Landscape ⟨ZZ⟩(θ₁, θ₂) — Estimator V2 vectorizado')

plt.tight_layout()
plt.show()

## 8. Primitivas con Qiskit Runtime (IBM Cloud)

En producción, se usa `QiskitRuntimeService` para ejecutar en hardware real.
Aquí mostramos el patrón completo (requiere credenciales IBM).

In [ ]:
# ── Patrón IBM Quantum Runtime (documentación, requiere credenciales) ──

runtime_pattern = """
# ── PATRÓN IBM QUANTUM RUNTIME (requiere cuenta IBM Quantum) ──────

from qiskit_ibm_runtime import QiskitRuntimeService, Session
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit_ibm_runtime.options import EstimatorOptions

# 1. Autenticación (una vez, guarda token en ~/.qiskit)
# QiskitRuntimeService.save_account(channel='ibm_quantum', token='MI_TOKEN')

# 2. Conectar al servicio y elegir backend
service = QiskitRuntimeService(channel='ibm_quantum')
backend = service.least_busy(operational=True, simulator=False, min_num_qubits=127)
print(f'Backend elegido: {backend.name}')

# 3. Transpilar al ISA del hardware
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
pm = generate_preset_pass_manager(optimization_level=3, backend=backend)
ansatz_isa = pm.run(ansatz)
H2_isa = H2_hamiltonian.apply_layout(ansatz_isa.layout)

# 4. Crear sesión y ejecutar (agrupa jobs en un único slot de QPU)
with Session(backend=backend) as session:
    # Opciones de mitigación de errores
    options = EstimatorOptions()
    options.resilience_level = 1          # M3 readout error mitigation
    options.optimization_level = 1

    estimator = Estimator(session=session, options=options)
    pub = (ansatz_isa, H2_isa, theta_optimo)
    job = estimator.run([pub])

    # job.job_id() para recuperar después
    print(f'Job ID: {job.job_id()}')
    result = job.result()

print(f'Energía en hardware real: {float(result[0].data.evs):.6f} Ha')

# ── Recuperar job posterior (si se guardó el job_id) ─────────────
# job_recuperado = service.job('JOB_ID_AQUI')
# result_recuperado = job_recuperado.result()
"""

print('Patrón IBM Quantum Runtime (requiere token):')  
print(runtime_pattern)

## 9. Gestión de sesiones y throttling

Las sesiones son críticas para algoritmos iterativos (VQE, QAOA) en hardware real.

In [ ]:
# ── Gestión de sesiones y throttling ──────────────────────────────

session_concepts = """
MODOS DE EJECUCIÓN EN IBM QUANTUM
═══════════════════════════════════════════════════════════════════

1. JOB MODE (sin sesión) — más común
   ─────────────────────────────────
   estimator.run([pub1, pub2, ...])
   → Cada llamada es un job independiente
   → Puede esperar en cola entre jobs
   → Mejor para tareas de un solo circuito

2. SESSION MODE — para algoritmos iterativos
   ──────────────────────────────────────────
   with Session(backend=backend) as session:
       estimator = EstimatorV2(session=session)
       for _ in range(n_iter):  # VQE, QAOA...
           job = estimator.run([pub])
           result = job.result()
   → Los jobs en la sesión tienen prioridad
   → La QPU se reserva mientras la sesión está abierta
   → Latencia reducida entre iteraciones
   → Cierre automático al salir del context manager

3. BATCH MODE — para múltiples circuitos no iterativos
   ────────────────────────────────────────────────────
   from qiskit_ibm_runtime import Batch
   with Batch(backend=backend) as batch:
       sampler = SamplerV2(session=batch)
       jobs = [sampler.run([pub]) for pub in pub_list]
   → Ejecuta todos los jobs en un bloque contíguo
   → Óptimo para barridos de parámetros

THROTTLING Y FAIR-SHARE
═══════════════════════════════════════════════════════════════════
• IBM usa fair-share scheduling: minutos de QPU por usuario
• Plan gratuito (Lite): 10 min/mes de QPU
• Plan Open: sin límite, pero colas largas
• Plan Premium: cuota dedicada
• sessions_max_time: tiempo máximo de una sesión (default 8h)
• max_execution_time por job: 300s default
"""

print(session_concepts)

In [ ]:
# Simulación de VQE iterativo con gestión de sesión (modo local)
# Demuestra el patrón correcto sin necesitar credenciales

import time

class LocalSession:
    """Simula el contexto de una sesión local para pruebas."""
    def __init__(self, backend_name: str = 'AerSimulator'):
        self.backend_name = backend_name
        self._active = False
        self.n_jobs = 0
        self.total_time_ms = 0

    def __enter__(self):
        self._active = True
        self._t_start = time.perf_counter()
        print(f'[Session] Abierta en {self.backend_name}')
        return self

    def __exit__(self, *_):
        elapsed = time.perf_counter() - self._t_start
        print(f'[Session] Cerrada. Jobs: {self.n_jobs}, Tiempo: {elapsed*1000:.1f} ms')
        self._active = False

    def run_job(self, estimator, pub):
        t0 = time.perf_counter()
        result = estimator.run([pub]).result()
        self.total_time_ms += (time.perf_counter() - t0) * 1000
        self.n_jobs += 1
        return result

# VQE con sesión local
energias_session = []
thetas_session = []

def cost_with_session(session, estimator_s, params):
    pub = (ansatz, H2_hamiltonian, np.array(params))
    result = session.run_job(estimator_s, pub)
    E = float(result[0].data.evs)
    energias_session.append(E)
    thetas_session.append(params[0])
    return E

with LocalSession('AerSimulator_local') as session:
    est_s = StatevectorEstimator()
    res_s = minimize(
        lambda p: cost_with_session(session, est_s, p),
        [0.2],
        method='COBYLA',
        options={'maxiter': 50}
    )

print(f'\nResultado VQE en sesión:')
print(f'  E_óptimo = {res_s.fun:.6f} Ha')
print(f'  θ_óptimo = {res_s.x[0]:.6f} rad')
print(f'  Iteraciones = {len(energias_session)}')

## 10. Comparativa Primitivas V1 vs V2

Qiskit 1.0 deprecó las primitivas V1. Aquí el mapa de migración.

In [ ]:
# ── Comparativa V1 vs V2 ──────────────────────────────────────────

comparativa = """
┌──────────────────────────────────────────────────────────────────────────────┐
│              PRIMITIVAS V1 → V2: GUÍA DE MIGRACIÓN                          │
├───────────────────┬─────────────────────────────┬──────────────────────────┤
│ Concepto          │ V1 (deprecado)               │ V2 (Qiskit ≥ 1.0)        │
├───────────────────┼─────────────────────────────┼──────────────────────────┤
│ Estimator import  │ from qiskit.primitives       │ from qiskit.primitives   │
│                   │   import Estimator           │   import StatevectorEst. │
│ Sampler import    │ from qiskit.primitives       │ from qiskit.primitives   │
│                   │   import Sampler             │   import StatevectorSam. │
├───────────────────┼─────────────────────────────┼──────────────────────────┤
│ Input (Estimator) │ .run(circuits, observables,  │ .run([(circuit, obs,     │
│                   │      parameters)             │        params), ...])    │
│ Input (Sampler)   │ .run(circuits, [shots])      │ .run([(circuit,),...],   │
│                   │                              │       shots=N)           │
├───────────────────┼─────────────────────────────┼──────────────────────────┤
│ Output (Estimator)│ result.values → np.ndarray   │ result[i].data.evs       │
│ Output (Sampler)  │ result.quasi_dists → dict    │ result[i].data.meas      │
│                   │                              │   (BitArray)             │
├───────────────────┼─────────────────────────────┼──────────────────────────┤
│ Param. batching   │ Manual (bucle)               │ Nativo: shape (N, n_par) │
│ Multi-PUB         │ Un circuito por job          │ Lista de PUBs por job    │
│ Error mitigation  │ Por kwargs separados         │ Integrado en Options     │
└───────────────────┴─────────────────────────────┴──────────────────────────┘
"""

print(comparativa)

# Demo de V1 → V2 migration para el mismo cálculo
# V2 (correcto)
estimator_v2 = StatevectorEstimator()
result_v2 = estimator_v2.run([
    (ansatz, H2_hamiltonian, theta_optimo)
]).result()
E_v2 = float(result_v2[0].data.evs)
print(f'V2 resultado: {E_v2:.6f} Ha')

## 11. QAOA con Qiskit Patterns

In [ ]:
# ── QAOA para MAX-CUT con Qiskit Patterns ─────────────────────────
# Grafo de 4 nodos: triángulo + extra
# Aristas: (0,1), (1,2), (2,3), (0,3)

edges = [(0, 1), (1, 2), (2, 3), (0, 3)]
n_nodes = 4

# ── Fase 1: Map ───────────────────────────────────────────────────
# Hamiltoniano de coste MAX-CUT: H_C = Σ_(i,j)∈E (I - Z_i Z_j) / 2
pauli_list = []
for i, j in edges:
    pauli_str_ii = 'I' * n_nodes
    pauli_str_zz = list('I' * n_nodes)
    pauli_str_zz[i] = 'Z'
    pauli_str_zz[j] = 'Z'
    pauli_list.append((''.join(pauli_str_ii), 0.5))
    pauli_list.append((''.join(pauli_str_zz), -0.5))

H_cost = SparsePauliOp.from_list(pauli_list)

# Circuito QAOA p=1
gamma = ParameterVector('γ', 1)
beta  = ParameterVector('β', 1)

qc_qaoa = QuantumCircuit(n_nodes)
qc_qaoa.h(range(n_nodes))        # estado inicial |+⟩^n

# Capa de coste: exp(-iγH_C)
for i, j in edges:
    qc_qaoa.cx(i, j)
    qc_qaoa.rz(2 * gamma[0], j)
    qc_qaoa.cx(i, j)

# Capa de mixer: exp(-iβH_B)
for q in range(n_nodes):
    qc_qaoa.rx(2 * beta[0], q)

print('QAOA (p=1) para MAX-CUT en 4 nodos:')
print(qc_qaoa.draw())

# ── Fase 3-4: Execute + Optimize ──────────────────────────────────
estimator_qaoa = StatevectorEstimator()
energias_qaoa = []

def cost_qaoa(params):
    g, b = params
    pub = (qc_qaoa, H_cost, np.array([g, b]))
    result = estimator_qaoa.run([pub]).result()
    E = -float(result[0].data.evs)  # maximizar corte
    energias_qaoa.append(-E)
    return E

res_qaoa = minimize(cost_qaoa, [0.5, 0.5], method='COBYLA',
                    options={'maxiter': 200})

gamma_opt, beta_opt = res_qaoa.x
max_cut_approx = -res_qaoa.fun

print(f'\nResultado QAOA MAX-CUT:')
print(f'  γ_opt = {gamma_opt:.4f}, β_opt = {beta_opt:.4f}')
print(f'  Corte aproximado: {max_cut_approx:.4f}')
print(f'  Corte exacto MAX-CUT (4 nodos, C4): 4.0')

In [ ]:
# Muestrear la solución QAOA óptima con Sampler

qc_qaoa_meas = qc_qaoa.assign_parameters(
    {gamma[0]: gamma_opt, beta[0]: beta_opt}
)
qc_qaoa_meas.measure_all()

sampler_qaoa = StatevectorSampler()
job_qaoa = sampler_qaoa.run([(qc_qaoa_meas,)], shots=5_000)
counts_qaoa = job_qaoa.result()[0].data.meas.get_counts()

# Calcular corte para cada bitstring
def calc_cut(bitstring: str, edges: list) -> int:
    bits = [int(b) for b in bitstring]
    return sum(1 for i, j in edges if bits[i] != bits[j])

# Top 10 medidas
print('Top 10 medidas QAOA (soluciones candidatas MAX-CUT):')
print(f'{"Bitstring":>10} | {"Corte":>6} | {"Frecuencia":>10} | {"P"}')
print('-' * 40)
for bstr, freq in sorted(counts_qaoa.items(), key=lambda x: -x[1])[:10]:
    corte = calc_cut(bstr, edges)
    p = freq / 5_000
    marker = ' ★' if corte == 4 else ''
    print(f'{bstr:>10} | {corte:>6} | {freq:>10} | {p:.3f}{marker}')

print('\n★ = corte máximo (4 aristas cortadas)')

# Gráfica de convergencia QAOA
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(energias_qaoa, 'b-', lw=1.5, alpha=0.8)
ax.axhline(4.0, color='red', ls='--', label='MAX-CUT óptimo = 4')
ax.set_xlabel('Iteración COBYLA')
ax.set_ylabel('Valor de corte ⟨C⟩')
ax.set_title('Convergencia QAOA MAX-CUT (p=1)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 12. Qiskit Serverless (patrón cloud-native)

Qiskit Serverless permite ejecutar código Python directamente en infraestructura IBM.

In [ ]:
# ── Qiskit Serverless: patrón cloud-native ────────────────────────

serverless_pattern = """
# Qiskit Serverless — ejecutar funciones en IBM Cloud
# Ventaja: el código Python corre cerca del hardware, latencia mínima

# ── Crear una QiskitFunction (se sube como programa) ──────────────
from qiskit_serverless import QiskitFunction, save_result, get_arguments

# Archivo: vqe_function.py (se sube a IBM Cloud)
"""Función VQE serverless."""
from qiskit_serverless import get_arguments, save_result
from qiskit_ibm_runtime import QiskitRuntimeService, EstimatorV2, Session
from scipy.optimize import minimize
import numpy as np

args = get_arguments()  # argumentos pasados al invocar
ansatz = args['ansatz']
hamiltonian = args['hamiltonian']
theta0 = args['theta0']

service = QiskitRuntimeService()
backend = service.least_busy(min_num_qubits=127, operational=True)

with Session(backend=backend) as session:
    estimator = EstimatorV2(session=session)
    energias = []

    def cost(params):
        pub = (ansatz, hamiltonian, np.array(params))
        E = float(estimator.run([pub]).result()[0].data.evs)
        energias.append(E)
        return E

    result = minimize(cost, theta0, method='COBYLA')

save_result({  # resultado accesible desde el cliente
    'E_optimo': result.fun,
    'theta_optimo': result.x.tolist(),
    'n_iter': len(energias),
    'convergencia': energias,
})

# ── Desde el cliente: subir y ejecutar ───────────────────────────
from qiskit_serverless import QiskitServerless, QiskitFunction
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
serverless = QiskitServerless(token=service.auth_token)

# Subir la función
func = QiskitFunction(
    title='vqe-h2',
    entrypoint='vqe_function.py',
    working_dir='./'
)
serverless.upload(func)

# Ejecutar con argumentos
job = serverless.run(
    'vqe-h2',
    arguments={'ansatz': ansatz, 'hamiltonian': H2_hamiltonian, 'theta0': [0.2]}
)

# Recuperar resultado (puede tardar minutos en hardware real)
print(f'Job ID: {job.job_id}')
result = job.result()  # bloquea hasta terminar
print(f'E_optimo = {result["E_optimo"]:.6f} Ha')
"""

print('Patrón Qiskit Serverless (requiere IBM Cloud):')
print(serverless_pattern)

## 13. Comparativa: Local vs Simulator vs Hardware real

In [ ]:
# ── Comparativa de backends ───────────────────────────────────────

from qiskit_aer.primitives import Estimator as AerEstimator
from qiskit_aer.noise import NoiseModel
import time

# 1. Local StatevectorEstimator (exacto, sin ruido)
t0 = time.perf_counter()
est_local = StatevectorEstimator()
res_local = est_local.run([(ansatz, H2_hamiltonian, theta_optimo)]).result()
E_local = float(res_local[0].data.evs)
t_local = time.perf_counter() - t0

# 2. Aer Estimator (con ruido de depolarizing)
from qiskit_aer.noise import depolarizing_error
noise_model = NoiseModel()
error_1q = depolarizing_error(0.001, 1)   # 0.1% por puerta de 1 qubit
error_2q = depolarizing_error(0.01, 2)    # 1% por puerta de 2 qubits
noise_model.add_all_qubit_quantum_error(error_1q, ['rx', 'ry', 'rz', 'h', 'x'])
noise_model.add_all_qubit_quantum_error(error_2q, ['cx', 'ecr', 'cz'])

t0 = time.perf_counter()
est_aer = AerEstimator()
est_aer.set_options(noise_model=noise_model, shots=10_000)
res_aer = est_aer.run(ansatz, H2_hamiltonian, theta_optimo).result()
E_aer = float(res_aer.values[0])
t_aer = time.perf_counter() - t0

# Tabla comparativa
print('╔══════════════════════════════════════════════════════════════════╗')
print('║         COMPARATIVA DE BACKENDS PARA VQE H₂                    ║')
print('╠════════════════════════╦══════════════╦════════════╦════════════╣')
print('║ Backend                ║ E (Ha)       ║ Error (Ha) ║ Tiempo     ║')
print('╠════════════════════════╬══════════════╬════════════╬════════════╣')
E_ref = -1.857300
print(f'║ Exacto FCI             ║ {E_ref:>12.6f} ║ {0.0:>10.6f} ║ —          ║')
print(f'║ StatevectorEstimator   ║ {E_local:>12.6f} ║ {abs(E_local-E_ref):>10.6f} ║ {t_local*1000:>6.1f} ms  ║')
print(f'║ Aer (ruido 1%)         ║ {E_aer:>12.6f} ║ {abs(E_aer-E_ref):>10.6f} ║ {t_aer*1000:>6.1f} ms  ║')
print(f'║ IBM Hardware (típico)  ║ ~ -1.84 Ha   ║ ~  0.02    ║ ~  2-5 min ║')
print('╚════════════════════════╩══════════════╩════════════╩════════════╝')
print('\n(Los tiempos de IBM incluyen tiempo en cola, que puede ser horas)')

## 14. Resumen: flujo completo de Qiskit Patterns

In [ ]:
# ── Resumen ejecutivo ─────────────────────────────────────────────

resumen = """
QISKIT PATTERNS — RESUMEN
═══════════════════════════════════════════════════════════════════════════

PRIMITIVAS V2
  StatevectorEstimator → ⟨O⟩ exacto sin ruido (local, rápido)
  StatevectorSampler   → distribución de medidas exacta
  AerEstimator         → ⟨O⟩ con modelo de ruido realista
  EstimatorV2 (Runtime)→ hardware IBM Quantum

FORMATO PUB (Primitive Unified Bloc)
  Estimator: (circuit, observables, parameter_values)
  Sampler:   (circuit,)  + shots=N en .run()
  → Siempre pasa una LISTA de PUBs a .run()
  → result[i] para acceder al i-ésimo PUB

ACCESO A RESULTADOS V2
  Estimator: result[0].data.evs          → np.ndarray de valores esperados
  Sampler:   result[0].data.meas         → BitArray
             .get_counts()               → dict[str, int]
             .get_int_counts()           → dict[int, int]
             .get_bitstrings()           → np.ndarray de strings

BATCHING DE PARÁMETROS (Estimator)
  param_values shape (N, n_params) → evs shape (N,)
  → Todos los N puntos en un solo PUB, no N llamadas a .run()

SESIONES (IBM Quantum)
  Session → algoritmos iterativos (VQE, QAOA)
  Batch   → barridos no iterativos de circuitos
  Sin sesión → jobs individuales (entra en cola cada vez)

QISKIT SERVERLESS
  → Código Python corre en IBM Cloud, cerca del hardware
  → Latencia de red mínima entre iteraciones del optimizador
  → Resultado accesible desde cliente con job.result()
"""
print(resumen)